# SUDO KG MCP HTTP Server Playground

Use this notebook when the MCP server is already running as a server. It does **not** spawn `src/server.py`; every cell connects to `MCP_ENDPOINT` over streamable HTTP.

Start the server in a separate terminal first, then run cells here.

## Start The Server

From the repo root, run:

```bash
export SKG_BACKEND=sudo_kg
export SKG_MCP_TRANSPORT=streamable-http
export SKG_MCP_HOST=127.0.0.1
export SKG_MCP_PORT=8000
export SKG_MCP_PATH=/mcp

uv run src/server.py
```

The SUDO backend also loads `.env`, so your Fuseki, Milvus, and LM Studio/OpenAI-compatible embedding settings can stay there.

In [1]:
import json
from typing import Any

from skg_mcp.client import SKGMCPClient
from skg_mcp.models import (
    ExpandContextArgs,
    ExpandNeighborsArgs,
    FilterPapersArgs,
    GetAttributionArgs,
    GetProvenanceArgs,
    LexicalSearchArgs,
    NodeKind,
    ResolveConceptReferenceArgs,
    SearchNodeType,
    SemanticConstraintSearchArgs,
    SemanticSearchArgs,
    SparqlTemplateFilter,
    StatementFilter,
)

MCP_ENDPOINT = "http://127.0.0.1:8004/mcp"

def pretty(obj: Any) -> None:
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump(mode="json")
    print(json.dumps(obj, indent=2, ensure_ascii=False))

MCP_ENDPOINT

'http://127.0.0.1:8004/mcp'

## Connect And Inspect

In [2]:
async with SKGMCPClient.connect_streamable_http(MCP_ENDPOINT, read_timeout_seconds=120) as client:
    metadata = await client.server_metadata()
    tools = await client.list_tools()

pretty(metadata)
print([tool.name for tool in tools])

  + Exception Group Traceback (most recent call last):
  |   File "/home/nipdep/Dev/skg-mcp/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3745, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/tmp/ipykernel_3259094/1533093002.py", line 1, in <module>
  |     async with SKGMCPClient.connect_streamable_http(MCP_ENDPOINT, read_timeout_seconds=120) as client:
  |                ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/home/nipdep/.local/share/uv/python/cpython-3.13-linux-aarch64-gnu/lib/python3.13/contextlib.py", line 214, in __aenter__
  |     return await anext(self.gen)
  |            ^^^^^^^^^^^^^^^^^^^^^
  |   File "/home/nipdep/Dev/skg-mcp/src/skg_mcp/client.py", line 71, in connect_streamable_http
  |     async with streamable_http_client(url) as (read_stream, write_stream, _):
  |                ~~~~~~~~~~~~~~~~~~~~~~^^^^^
  |   File "/home/nipdep/.local/share/uv/py

## Shared State

Run top-to-bottom once to populate seed IDs, then edit these variables manually for focused debugging.

In [ ]:
QUERY = "eventuality entailment graph"
paper_id = None
concept_id = None
concept_label = None
statement_id = None
statement_text = None

QUERY

'eventuality entailment graph'

## `filter_papers`

In [ ]:
async with SKGMCPClient.connect_streamable_http(MCP_ENDPOINT, read_timeout_seconds=120) as client:
    filter_papers_result = await client.filter_papers(FilterPapersArgs(limit=5))

pretty(filter_papers_result)
if filter_papers_result.papers:
    paper_id = filter_papers_result.papers[0].id
paper_id

{
  "papers": [
    {
      "id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
      "title": "Enriching Large-Scale Eventuality Knowledge Graph with Entailment Relations",
      "year": null,
      "venue": null
    },
    {
      "id": "revisiting_evaluation_of_knowledge_base_completion_models",
      "title": "Revisiting Evaluation of Knowledge Base Completion Models",
      "year": null,
      "venue": null
    },
    {
      "id": "learning_credal_sum_product_networks",
      "title": "Learning Credal Sum Product Networks",
      "year": null,
      "venue": null
    },
    {
      "id": "syntactic_question_abstraction_and_retrieval_for_data_scarce_semantic_parsing",
      "title": "Syntactic Question Abstraction and Retrieval for Data-Scarce Semantic Parsing",
      "year": null,
      "venue": null
    },
    {
      "id": "ranking_vs_classifying_measuring_knowledge_base_completion_quality",
      "title": "Ranking vs. Classifying: Measuring Kno

'enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations'

## `lexical_search`

In [ ]:
async with SKGMCPClient.connect_streamable_http(MCP_ENDPOINT, read_timeout_seconds=120) as client:
    lexical_result = await client.lexical_search(
        LexicalSearchArgs(
            query=QUERY,
            node_types=[SearchNodeType.CONCEPT, SearchNodeType.STATEMENT],
            limit=6,
        )
    )

pretty(lexical_result)
if lexical_result.concepts:
    concept_id = lexical_result.concepts[0].id
    concept_label = lexical_result.concepts[0].label
if lexical_result.statements:
    statement_id = lexical_result.statements[0].id
    statement_text = lexical_result.statements[0].text

concept_id, concept_label, statement_id

{
  "concepts": [
    {
      "id": "4ce42fca-be73-4a58-a28b-ec232acb77b0",
      "label": "three-step eventuality entailment graph construction method",
      "aliases": null,
      "concept_type": "Method",
      "is_canonical": false,
      "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
      "score": null,
      "summary": null,
      "provenance": null
    },
    {
      "id": "0d08976a-6701-4aa1-a17b-e403c32c4d1b",
      "label": "eventuality entailment graph construction task",
      "aliases": null,
      "concept_type": "Artifact",
      "is_canonical": false,
      "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
      "score": null,
      "summary": null,
      "provenance": null
    }
  ],
  "statements": [
    {
      "id": "127229c1-9c7f-4f68-9e75-0a8730b9b925",
      "text": "large-scale eventuality entailment graph",
      "statement_type": "artifact",
      "rhetorical_role": "artifac

('4ce42fca-be73-4a58-a28b-ec232acb77b0',
 'three-step eventuality entailment graph construction method',
 '127229c1-9c7f-4f68-9e75-0a8730b9b925')

## `semantic_search`

In [ ]:
QUERY = "eventuality graph"

async with SKGMCPClient.connect_streamable_http(MCP_ENDPOINT, read_timeout_seconds=120) as client:
    semantic_result = await client.semantic_search(
        SemanticSearchArgs(
            query=QUERY,
            # node_types=[SearchNodeType.CONCEPT, SearchNodeType.STATEMENT],
            limit=10,
        )
    )

pretty(semantic_result)

{
  "concepts": [
    {
      "id": "1ad0282d-9cf8-4132-9a8a-db4793fad482",
      "label": "complicated eventuality graph",
      "aliases": null,
      "concept_type": "Term",
      "is_canonical": false,
      "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
      "score": null,
      "summary": null,
      "provenance": null
    },
    {
      "id": "b6e372cc-cb15-4ce6-b3ce-f4ca386ab51f",
      "label": "eventuality graph",
      "aliases": null,
      "concept_type": "Term",
      "is_canonical": false,
      "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
      "score": null,
      "summary": null,
      "provenance": null
    }
  ],
  "statements": [
    {
      "id": "80b8c3d6-b9c1-40af-8940-48a34f1f3be8",
      "text": "On top of the decomposed eventuality representation, we can then conduct local and global entailment inferences to construct the eventuality graph.",
      "statement_type": "Ap

## `semantic_constraint_search` With SPARQL Template Filter

In [ ]:
concept_iri = f"https://w3id.org/twc/sudo/kg/concept/{concept_id}" if concept_id else None

statement_filters = None
if concept_iri:
    statement_filters = StatementFilter(
        sparql_filters=[
            SparqlTemplateFilter(
                where="?node sudo:mentions ?concept . FILTER(?concept IN ({{concepts}}))",
                params={"concepts": [concept_iri]},
                graph="sudo",
            )
        ]
    )

async with SKGMCPClient.connect_streamable_http(MCP_ENDPOINT, read_timeout_seconds=120) as client:
    constrained_result = await client.semantic_constraint_search(
        SemanticConstraintSearchArgs(
            query=QUERY,
            node_types=[SearchNodeType.STATEMENT],
            statement_filters=statement_filters,
            limit=5,
        )
    )

pretty(constrained_result)

{
  "concepts": [],
  "statements": []
}


## `resolve_concept_reference`

In [ ]:
mention = concept_label or QUERY
context_text = statement_text or QUERY

async with SKGMCPClient.connect_streamable_http(MCP_ENDPOINT, read_timeout_seconds=120) as client:
    resolution_result = await client.resolve_concept_reference(
        ResolveConceptReferenceArgs(
            mention=mention,
            context_text=context_text,
            paper_id=paper_id,
            limit=3,
        )
    )

pretty(resolution_result)

{
  "resolutions": []
}


## Seed Node For Expansion/Provenance

In [ ]:
node_id = concept_id or statement_id
node_kind = NodeKind.CONCEPT if concept_id else NodeKind.STATEMENT

node_id, node_kind, paper_id

('4ce42fca-be73-4a58-a28b-ec232acb77b0',
 <NodeKind.CONCEPT: 'concept'>,
 'enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations')

## `expand_context`

In [ ]:
async with SKGMCPClient.connect_streamable_http(MCP_ENDPOINT, read_timeout_seconds=120) as client:
    context_result = await client.expand_context(
        ExpandContextArgs(
            node_id=node_id,
            node_kind=node_kind,
            paper_id=paper_id,
            include_artifacts=True,
            include_propositions=True,
            include_relations=True,
            limit=10,
        )
    )

pretty(context_result)

{
  "node": {
    "id": "4ce42fca-be73-4a58-a28b-ec232acb77b0",
    "kind": "concept",
    "label": "three-step eventuality entailment graph construction method",
    "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
    "score": null
  },
  "artifacts": [],
  "propositions": [
    {
      "id": "02c30181-327c-4cf0-923a-e89042f5a088",
      "kind": "statement",
      "label": "In this paper, to address the limitations of existing approaches, we propose a three-step eventuality entailment graph construction method.",
      "node_type": "Approach",
      "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
      "relation": "supports"
    }
  ]
}


## `expand_neighbors`

In [ ]:
async with SKGMCPClient.connect_streamable_http(MCP_ENDPOINT, read_timeout_seconds=120) as client:
    neighbors_result = await client.expand_neighbors(
        ExpandNeighborsArgs(
            node_id=node_id,
            node_kind=node_kind,
            paper_id=paper_id,
            hop_count=3,
            limit=10,
        )
    )

pretty(neighbors_result)

{
  "source_node": {
    "id": "4ce42fca-be73-4a58-a28b-ec232acb77b0",
    "kind": "concept",
    "label": "three-step eventuality entailment graph construction method",
    "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
    "score": null
  },
  "hop_count": 3,
  "neighbors": [
    {
      "relation_type": "supports",
      "target_id": "02c30181-327c-4cf0-923a-e89042f5a088",
      "target_kind": "statement",
      "target_label": "In this paper, to address the limitations of existing approaches, we propose a three-step eventuality entailment graph construction method.",
      "score": null,
      "hop": 1
    }
  ]
}


## `get_attribution`

In [ ]:
async with SKGMCPClient.connect_streamable_http(MCP_ENDPOINT, read_timeout_seconds=120) as client:
    attribution_result = await client.get_attribution(
        GetAttributionArgs(
            node_ids=[node_id],
            node_kind=node_kind,
            # paper_id=paper_id,
        )
    )

pretty(attribution_result)

{
  "attributions": [
    {
      "node_id": "4ce42fca-be73-4a58-a28b-ec232acb77b0",
      "paper": {
        "id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
        "title": "Enriching Large-Scale Eventuality Knowledge Graph with Entailment Relations",
        "year": null,
        "venue": null
      },
      "location": {
        "section_id": "b936da1c-3c78-49ff-a786-53a22b84584c",
        "section_title": "Introduction",
        "paragraph_id": "8db20798-a2d6-4636-bd0f-88ecf9c7ffce",
        "sentence_id": "423148de-ae52-49fe-a5b9-cb0a12e7fcbc",
        "sentence_text": null
      }
    }
  ]
}


## `get_provenance`

In [ ]:
async with SKGMCPClient.connect_streamable_http(MCP_ENDPOINT, read_timeout_seconds=120) as client:
    provenance_result = await client.get_provenance(
        GetProvenanceArgs(
            node_ids=[node_id],
            node_kind=node_kind,
            # paper_id=paper_id,
        )
    )

pretty(provenance_result)

{
  "provenance": [
    {
      "node_id": "4ce42fca-be73-4a58-a28b-ec232acb77b0",
      "provenance": [
        {
          "paper_id": "enriching_large_scale_eventuality_knowledge_graph_with_entailment_relations",
          "sentence_id": "423148de-ae52-49fe-a5b9-cb0a12e7fcbc"
        }
      ]
    }
  ]
}
